# 03 — Descobrindo perfis: K-means, elbow e PCA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/03_aprendizado_nao_supervisionado.ipynb)

**Duração estimada:** 60–75 minutos  
**Pré-requisitos:** Notebook 01 e noções de distância.

## Objetivos

- padronizar atributos e aplicar K-means
- comparar elbow e silhouette
- projetar os grupos em dois componentes principais
- caracterizar clusters sem chamá-los de fenótipos clínicos

## Fonte e licença

[CDC Diabetes Health Indicators — descrição das variáveis](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators), conjunto 891 da UCI. Consulte a tabela de variáveis para interpretar os códigos dos atributos, como 0 e 1. O alvo será excluído do agrupamento.

Fonte UCI sob CC BY 4.0; cite o conjunto e sua publicação.

> **Uso responsável:** Este material tem finalidade exclusivamente educacional. Os resultados não devem ser usados para diagnóstico, prognóstico, tratamento, gestão assistencial ou decisão de saúde pública sem validação adequada, análise de contexto e supervisão de profissionais qualificados.

## Onde executar

### Google Colab

Use o botão **Open In Colab** no início do notebook e escolha **Executar tudo**. A célula de preparação clona ou atualiza o repositório em `/content`, instala somente as dependências ausentes e fixa a semente aleatória.

### Computador local

Requisitos: Git e Python 3.10–3.13. No terminal, clone o projeto e crie um ambiente virtual:

```bash
git clone https://github.com/flavioluizseixas/aprendizado-de-maquina-para-saude.git
cd aprendizado-de-maquina-para-saude
python -m venv .venv
```

Ative-o no Windows PowerShell com `.\.venv\Scripts\Activate.ps1` ou, no Linux/macOS, com `source .venv/bin/activate`.

Instale somente as dependências deste encontro e abra o notebook a partir da raiz do repositório:

```bash
python -m pip install -e ".[clustering]"
jupyter lab notebooks/03_aprendizado_nao_supervisionado.ipynb
```

Não é necessário alterar caminhos nem fazer upload de arquivos. Fora do Colab, a próxima célula usa o repositório local e o mesmo ambiente Python selecionado como kernel do Jupyter.

## Preparação do ambiente

> Como garantir a mesma inicialização?

In [ ]:
# Preparação reproduzível do ambiente (a instalação ocorre só se faltar pacote).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "flavioluizseixas/aprendizado-de-maquina-para-saude"
REPO_DIR = Path("/content") / REPO.split("/")[-1]
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    command = ["git", "clone", f"https://github.com/{REPO}.git", str(REPO_DIR)]
    if REPO_DIR.exists():
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    subprocess.run(command, check=True)
    os.chdir(REPO_DIR)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    project = next((p for p in candidates if (p / "src").exists()), Path.cwd())
    os.chdir(project)

packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<4', 'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1', 'sklearn': 'scikit-learn>=1.4,<2', 'requests': 'requests>=2.31,<3', 'ucimlrepo': 'ucimlrepo>=0.0.7,<1', 'kneed': 'kneed>=0.8,<1'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from src.config import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)
print(f"Ambiente pronto em {Path.cwd()} | Colab={IN_COLAB} | semente={RANDOM_STATE}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from src.clustering import choose_k, evaluate_kmeans_range
from src.data_loading import load_cdc_diabetes

FAST_MODE = True
attributes = ["BMI", "Age", "GenHlth", "PhysHlth", "MentHlth", "Education", "Income"]
feature_labels = {
    "BMI": "IMC",
    "Age": "Faixa etária (1–13)",
    "GenHlth": "Saúde geral (1 excelente–5 ruim)",
    "PhysHlth": "Saúde física ruim (dias/30)",
    "MentHlth": "Saúde mental ruim (dias/30)",
    "Education": "Escolaridade (1–6)",
    "Income": "Faixa de renda (1–8)",
}
TARGET_NAME = "Indicador de diabetes"
target_labels = {0: "0 Sem diabetes", 1: "1 Pré-diabetes/diabetes"}
data, metadata = load_cdc_diabetes(
    15_000 if FAST_MODE else 40_000, random_state=RANDOM_STATE
)

### Dicionário das variáveis usadas

O agrupamento usa sete atributos e exclui `Diabetes_binary`. `Age`, `GenHlth`, `Education` e `Income` são códigos ordinais; as distâncias entre níveis são uma simplificação analítica. O desfecho só será consultado depois, para descrição externa.

In [ ]:
coding_summary = pd.DataFrame({
    "atributo_no_notebook": list(feature_labels.values()) + [TARGET_NAME],
    "nome_original": attributes + ["Diabetes_binary"],
    "interpretação": [
        "Índice de massa corporal",
        "Faixa etária ordenada de 1 a 13",
        "1=excelente; 2=muito boa; 3=boa; 4=regular; 5=ruim",
        "Número de dias de saúde física ruim nos últimos 30 dias",
        "Número de dias de saúde mental ruim nos últimos 30 dias",
        "Faixa de escolaridade ordenada de 1 a 6",
        "Faixa de renda ordenada de 1 a 8",
        "0=sem diabetes; 1=pré-diabetes ou diabetes (avaliação externa)",
    ],
})
display(coding_summary.style.hide(axis="index"))

variable_dictionary = metadata.get("variables")
if isinstance(variable_dictionary, pd.DataFrame) and "name" in variable_dictionary:
    selected_dictionary = variable_dictionary[
        variable_dictionary["name"].isin(attributes + ["Diabetes_binary"])
    ].copy()
    available = [
        column for column in ["name", "role", "type", "description", "units"]
        if column in selected_dictionary
    ]
    display(selected_dictionary[available].rename(columns={
        "name": "nome_original", "role": "papel", "type": "tipo",
        "description": "descrição_UCI", "units": "unidade",
    }))

## Pergunta orientadora

> Existem agrupamentos matemáticos estáveis nesses indicadores, e como descrevê-los sem transformá-los em classes clínicas?

## Inspeção e preparação

> O alvo entrou por engano no agrupamento? As escalas são comparáveis?

In [ ]:
assert "Diabetes_binary" not in attributes
X = data[attributes].rename(columns=feature_labels)
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(imputer.fit_transform(X))
print("Matriz do agrupamento:", X_scaled.shape, "| alvo excluído:", "Diabetes_binary" not in attributes)

## Experimento: quantos clusters?

> O cotovelo e o silhouette apontam para a mesma escolha?

In [ ]:
results = evaluate_kmeans_range(X_scaled, range(2, 9), random_state=RANDOM_STATE)
chosen_k, criterion = choose_k(results)
display(results.round(3))
print(f"k escolhido = {chosen_k} ({criterion})")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
results.plot(x="k", y="inércia", marker="o", ax=axes[0], legend=False, title="Elbow")
results.plot(x="k", y="silhouette", marker="o", ax=axes[1], legend=False, title="Silhouette")
plt.tight_layout(); plt.show()

### Como interpretar

Inércia sempre cai quando k cresce; buscamos uma mudança de inclinação. Silhouette favorece grupos compactos e separados. A escolha continua sendo uma decisão analítica, não uma descoberta de doenças.

## PCA e visualização

> Quanta informação dois eixos preservam?

In [ ]:
kmeans = KMeans(n_clusters=chosen_k, n_init=10, random_state=RANDOM_STATE)
clusters = kmeans.fit_predict(X_scaled)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coordinates = pca.fit_transform(X_scaled)
centers_2d = pca.transform(kmeans.cluster_centers_)
print("Variância explicada:", pca.explained_variance_ratio_.round(3), "| total:", pca.explained_variance_ratio_.sum().round(3))

In [ ]:
plot_data = pd.DataFrame(coordinates, columns=["PC1", "PC2"])
plot_data["Cluster"] = [f"Cluster {value}" for value in clusters]
sample = plot_data.sample(min(5_000, len(plot_data)), random_state=RANDOM_STATE)
sns.scatterplot(data=sample, x="PC1", y="PC2", hue="Cluster", alpha=0.45, s=18, palette="tab10")
plt.scatter(centers_2d[:, 0], centers_2d[:, 1], marker="X", s=180, c="black", label="centróides")
plt.title(f"PCA dos clusters (amostra visual; n total={len(data):,})")
plt.legend(); plt.show()

## Avaliação e perfis

> O que caracteriza cada grupo na escala original e padronizada?

In [ ]:
profiled = X.copy()
profiled[TARGET_NAME] = data["Diabetes_binary"].to_numpy()
profiled["Cluster"] = [f"Cluster {value}" for value in clusters]
profile = profiled.groupby("Cluster")[list(X.columns)].mean()
prevalence = profiled.groupby("Cluster")[TARGET_NAME].agg(["mean", "count"])
prevalence["mean"] *= 100
display(profile.round(2))
display(prevalence.rename(columns={"mean": "prevalência externa (%)", "count": "n"}).round(1))
standardized_profile = (
    pd.DataFrame(X_scaled, columns=X.columns)
    .assign(Cluster=[f"Cluster {value}" for value in clusters])
    .groupby("Cluster").mean()
)
sns.heatmap(standardized_profile, cmap="vlag", center=0, annot=True, fmt=".1f")
plt.title("Perfil médio padronizado por cluster"); plt.show()

### Como interpretar

O alvo foi consultado **depois** apenas como avaliação externa. Diferenças de prevalência não tornam o cluster um diagnóstico. PCA perde informação, portanto separação ou sobreposição no plano não resume todo o espaço.

## Limitações e responsabilidade

- K-means depende de escala, inicialização, forma aproximadamente esférica e escolha de k.
- PCA com dois componentes omite parte da variabilidade.
- Clusters são construções matemáticas nesta amostra; chamá-los de fenótipos exigiria validação clínica externa.

## Atividade

Remova um atributo, repita elbow/silhouette e compare os perfis. Explique por que a solução mudou ou permaneceu semelhante.

## Três aprendizados principais

1. Padronizar evita que uma unidade domine as distâncias.
2. Elbow e silhouette oferecem evidências complementares, não uma verdade única.
3. O alvo só pode entrar depois, como descrição externa.

## Referências

- [UCI — CDC Diabetes Health Indicators](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)
- [scikit-learn — clustering](https://scikit-learn.org/stable/modules/clustering.html)
- [scikit-learn — PCA](https://scikit-learn.org/stable/modules/decomposition.html#pca)

## Versões das bibliotecas

Registre o ambiente junto ao resultado.

In [ ]:
from src.config import library_versions
library_versions(('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'kneed', 'ucimlrepo'))